i've done a similar task during the robocon season so hopefully i don't need to search much here.
i'll still follow 

https://mansoormemon.medium.com/implementing-object-detection-based-on-color-f49979814c6e



because i forgot the exact order of operations 




In [23]:
import cv2
import glob
from pathlib import Path
import numpy as np 
import os

def clean_mask(mask , kernel_size=13):
    kernel = np.ones((kernel_size, kernel_size), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    return mask

## Loading


In [24]:
current_dir = Path.cwd()
folder_path = Path.joinpath(current_dir , "balls")
images_path = glob.glob(f"{folder_path}/*.jpg")
print (images_path)


images = [cv2.imread(path) for path in images_path]

['/home/youssef/mia_ai/Task_5/Task5.2/balls/ball_7.jpg', '/home/youssef/mia_ai/Task_5/Task5.2/balls/ball_10.jpg', '/home/youssef/mia_ai/Task_5/Task5.2/balls/ball_15.jpg', '/home/youssef/mia_ai/Task_5/Task5.2/balls/ball_12.jpg', '/home/youssef/mia_ai/Task_5/Task5.2/balls/ball_14.jpg', '/home/youssef/mia_ai/Task_5/Task5.2/balls/ball_3.jpg', '/home/youssef/mia_ai/Task_5/Task5.2/balls/ball_16.jpg', '/home/youssef/mia_ai/Task_5/Task5.2/balls/ball_20.jpg', '/home/youssef/mia_ai/Task_5/Task5.2/balls/ball_4.jpg', '/home/youssef/mia_ai/Task_5/Task5.2/balls/ball_17.jpg', '/home/youssef/mia_ai/Task_5/Task5.2/balls/ball_13.jpg', '/home/youssef/mia_ai/Task_5/Task5.2/balls/ball_2.jpg', '/home/youssef/mia_ai/Task_5/Task5.2/balls/ball_11.jpg', '/home/youssef/mia_ai/Task_5/Task5.2/balls/ball_1.jpg', '/home/youssef/mia_ai/Task_5/Task5.2/balls/ball_19.jpg', '/home/youssef/mia_ai/Task_5/Task5.2/balls/ball_8.jpg', '/home/youssef/mia_ai/Task_5/Task5.2/balls/ball_5.jpg', '/home/youssef/mia_ai/Task_5/Task5.2/

## Preprocessing 


In [25]:
#bgr to hsv since representing colors is easier there using value 
hsv_images = [cv2.cvtColor(image,cv2.COLOR_BGR2HSV) for image in images]

#i had hsv parameter tuner built already for robocon , i'll implement something similar if this ranges fail
LOWER_RED_1 = np.array([0, 64, 16])
UPPER_RED_1 = np.array([10, 255, 255])
LOWER_RED_2 = np.array([170, 64, 16])
UPPER_RED_2 = np.array([180, 255, 255])

LOWER_BLUE = np.array([100, 64, 16])
UPPER_BLUE = np.array([130, 255, 255])

mask_red_1 = [cv2.inRange(hsv_image , LOWER_RED_1 , UPPER_RED_1) for hsv_image in hsv_images] 
mask_red_2 = [cv2.inRange(hsv_image , LOWER_RED_2 , UPPER_RED_2) for hsv_image in hsv_images] 
mask_blue  = [cv2.inRange(hsv_image , LOWER_BLUE , UPPER_BLUE) for hsv_image in hsv_images] 

#grouping masks for each image
mask_red   = [cv2.bitwise_or(r1, r2) for r1, r2 in zip(mask_red_1, mask_red_2)] 

#applying dilation and erosion
mask_red  = [clean_mask(mask) for mask in mask_red]
mask_blue = [clean_mask(mask)for mask in mask_blue]

os.makedirs("output_labeled", exist_ok=True)

MIN_AREA = 324
MIN_CIRCULARITY = 0.2
COLOR_SPECS = [
    (mask_red,  (0, 0, 255), "red"),
    (mask_blue, (255, 0, 0), "blue"),
]

def is_circular(contour):
    area = cv2.contourArea(contour)
    if area <= MIN_AREA:
        return False
    perimeter = cv2.arcLength(contour, True)
    if perimeter == 0:
        return False
    circularity = 4 * np.pi * area / (perimeter ** 2)
    return circularity > MIN_CIRCULARITY

for i, image in enumerate(images):
    frame = image.copy()

    for mask_list, box_color, label in COLOR_SPECS:
        contours, _ = cv2.findContours(mask_list[i], cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
        filtered_contours = [c for c in contours if is_circular(c)]

        for c in filtered_contours:
            x, y, w, h = cv2.boundingRect(c)
            cv2.rectangle(frame, (x, y), (x + w, y + h), box_color, 1)
            cv2.putText(frame, label, (x, y - 5), cv2.FONT_HERSHEY_SIMPLEX,
                        0.5, box_color, 1)

    cv2.imwrite(f"output_labeled/frame{i}.jpg", frame)